# Julia notebook to compute the symbolic quadratures for the solution of the robust diagnostic PG problem
twnh 

Aug '26.  

This notebook reads the output from `BuoyancyEquationSolution.ipynb` and `FrictionalGeostrophicEquationSolution.ipynb`, namely the Green's functions $G_{\mathfrak{u}}$ and $G_b$ and computes integrals using them. It then writes out the following expression for later use.

\begin{align}
G_{ \mathfrak{u}, b} (z; \xi') & = 
\int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z'; \xi') \; dz'  \; d \xi .
\end{align}

See `SymbolicsTesting.jl` to test this result against numerical quadrature.

The notebook makes a plot of this $G_{ \mathfrak{u}, b} (z; \xi')$ function.

The notebook also computes 

\begin{align}
G_{ \mathfrak{u}, {b_\xi}} (z) & = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi ,\\
G_{ \mathfrak{u}_y, b}& = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z';\xi)   \; dz'  \; d \xi , \\
G_{ \mathfrak{u}_y, {b_\xi}} (z) & = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi .
\end{align}

In [ ]:
using SymPy
using Symbolics
using JLD2

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",    real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
zp     = symbols("zp",     real=true, negative=true)
ξp     = symbols("ξp",     real=true, negative=true)
x, y   = symbols("x y",    real=true)                # Horizontal coordinates
H      = symbols("H",      real=true, positive=true) # Domain depth

ψ      = symbols("ψ",      real=true, positive=true) 
φ      = symbols("φ", imaginary=true)

κ      = symbols("κ",      real=true, positive=true) # Diffusivity parameter
ν      = symbols("ν",      real=true, positive=true) # Viscosity parameters

Ek     = symbols("Ek",    real=true, positive=true) # Ekman number
f      = SymFunction("f", real=true)                # Coriolis parameter
β      = symbols("β",      real=true, positive=true) # Planetary vorticity gradient

# Dictionary for parsing strings.
locals = Dict(
    "x" => x,
    "y" => y,
    "z" => z,
    "ξ" => ξ,
    "κ" => κ,
    "ν" => ν,
    "φ" => φ,
    "ψ" => ψ,
    "H" => H
    )

# Parse expressions using the canonical symbols.
Guv_str = read("sympy_Guv_expression.txt", String)
Guv = sympy.sympify(Guv_str, locals=locals)
println("0.1 Guv expression loaded from file:")
display(Guv)
println()

Gb_str  = read("sympy_Gb_expression.txt", String)
Gb  = sympy.sympify(Gb_str,  locals=locals)
println("0.2 Gb expression loaded from file:")
display(Gb)
println()

#### Compute $ \int_{\xi}^{0}  G_b(z';\xi ')  \; dz' $

In [ ]:

println()
println("1.1 Symbolic computation of the integral of Gb...")
@time begin
    tmp_Gb = sympy.expand(Gb.subs(Dict(z => zp, ξ => ξp)))
    inner_Gb_integral = sympy.integrate(tmp_Gb, (zp, ξ, 0))
    max_obj = sympy.Max(ξ,ξp)
    min_obj = sympy.Min(0, max_obj)
    inner_Gb_integral = inner_Gb_integral.subs(min_obj,max_obj)
    # Create two explicit branches
    expr_if_xi_ge_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp), ξ)
    expr_if_xi_lt_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp),ξp)

    # Form a piecewise object
    inner_Gb_integral = sympy.Piecewise(
    (expr_if_xi_ge_xip, sympy.Ge(ξ, ξp)),
    (expr_if_xi_lt_xip, sympy.Lt(ξ, ξp)))
end

println("1.1 Done.")
# display(inner_Gb_integral)

#### Compute $ \int_{-H}^{0}$ Guv $(z; \xi) \, $ inner_Gb_integral $(\xi, \xi ') \; d \xi $

In [ ]:
function integrate_Guv_Gb(Guv_branch1, Guv_branch2, inner_Gb_integral)
    println("  integrate_Guv_Gb: Starting integration of f(Guv) and g(Gb).")
    # ------------------------------------------------------------------
    # Extract branch expressions
    #
    # Guv:
    #   Guv_branch1: z ≤ ξ  ⇔ ξ ≥ z
    #   Guv_branch2: z ≥ ξ  ⇔ ξ ≤ z
    #
    # inner_Gb_integral:
    #   Gbint_branch1: ξ ≥ ξp
    #   Gbint_branch2: ξ <  ξp
    # ------------------------------------------------------------------
    Gbint_branch1 = inner_Gb_integral.args[1].args[1]   # ξ ≥ ξp
    Gbint_branch2 = inner_Gb_integral.args[2].args[1]   # ξ <  ξp

    # ------------------------------------------------------------------
    # Products
    # ------------------------------------------------------------------
    f11 = sympy.simplify(Guv_branch1 * Gbint_branch1)  # ξ ≥ z, ξ ≥ ξp
    f12 = sympy.simplify(Guv_branch1 * Gbint_branch2)  # ξ ≥ z, ξ <  ξp
    f21 = sympy.simplify(Guv_branch2 * Gbint_branch1)  # ξ ≤ z, ξ ≥ ξp
    f22 = sympy.simplify(Guv_branch2 * Gbint_branch2)  # ξ ≤ z, ξ <  ξp

    # ------------------------------------------------------------------
    # Case A: z ≤ ξp
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ z ≤ ξp ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, z]     → ξ ≤ z and ξ < ξp   → f22
    #   ξ ∈ [z, ξp]     → ξ ≥ z and ξ < ξp   → f12
    #   ξ ∈ [ξp, 0]     → ξ ≥ z and ξ ≥ ξp   → f11
    # ------------------------------------------------------------------
    I_A_22 = sympy.integrate(f22, (ξ, -H, z))
    println("  integrate_Guv_Gb: Computed I_A_22")
    I_A_12 = sympy.integrate(f12, (ξ,  z, ξp))
    println("  integrate_Guv_Gb: Computed I_A_12")
    I_A_11 = sympy.integrate(f11, (ξ, ξp, 0))
    println("  integrate_Guv_Gb: Computed I_A_11")

    I_A = sympy.simplify(I_A_22 + I_A_12 + I_A_11)

    # ------------------------------------------------------------------
    # Case B: ξp < z
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ ξp < z ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, ξp]    → ξ < ξp and ξ ≤ z   → f22
    #   ξ ∈ [ξp, z]     → ξ ≥ ξp and ξ ≤ z   → f21
    #   ξ ∈ [z, 0]      → ξ ≥ ξp and ξ ≥ z   → f11
    # ------------------------------------------------------------------
    I_B_22 = sympy.integrate(f22, (ξ, -H, ξp))
    println("  integrate_Guv_Gb: Computed I_B_22")
    I_B_21 = sympy.integrate(f21, (ξ, ξp, z))
    println("  integrate_Guv_Gb: Computed I_B_21")
    I_B_11 = sympy.integrate(f11, (ξ,  z, 0))
    println("  integrate_Guv_Gb: Computed I_B_11")

    I_B = sympy.simplify(I_B_22 + I_B_21 + I_B_11)

    # ------------------------------------------------------------------
    # Final piecewise outer integral
    # ------------------------------------------------------------------
    return sympy.Piecewise(
        (I_A, sympy.Le(z, ξp)),
        (I_B, sympy.Gt(z, ξp))
    )
end

#### Now compute $G_{ \mathfrak{u}, b} = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} G_b(z';\xi)  \; dz'  \; d \xi $

In [ ]:
function get_sympy_symbol(expr, name)
    syms = collect(expr.free_symbols)
    matches = [s for s in syms if string(s) == name]
    @assert length(matches) == 1 "Could not uniquely find SymPy symbol $name. Found: $matches"
    return matches[1]
end

function integrate_Guv_times_nonpiecewise(Guv, inner_Gb2_integral)
    println("  Starting split integration of Guv * inner_Gb2_integral.")

    expr = Guv * inner_Gb2_integral

    ξs = get_sympy_symbol(expr, "ξ")
    zs = get_sympy_symbol(expr, "z")
    Hs = get_sympy_symbol(expr, "H")

    Guv_branch1 = Guv.args[1].args[1]   # z ≤ ξ, i.e. ξ ≥ z
    Guv_branch2 = Guv.args[2].args[1]   # z ≥ ξ, i.e. ξ ≤ z

    println("  Integrating lower interval [-H, z]...")
    I_low = sympy.integrate(Guv_branch2 * inner_Gb2_integral, (ξs, -Hs, zs))
    println("  Done lower interval.")

    println("  Integrating upper interval [z, 0]...")
    I_high = sympy.integrate(Guv_branch1 * inner_Gb2_integral, (ξs, zs, 0))
    println("  Done upper interval.")

    println("  Combining.")
    return sympy.cancel(I_low + I_high)
end

In [ ]:
println()
println("1.2 Symbolic computation of Guvb...")
@time begin
    Guv_branch1 = Guv.args[1].args[1]                    # z ≤ ξ
    Guv_branch2 = Guv.args[2].args[1]                    # z ≥ ξ
    Guvb = integrate_Guv_Gb(Guv_branch1, Guv_branch2, inner_Gb_integral)
end

println("1.2 Done.")
# display(Guvb)

#### compute $ \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz' $

In [ ]:
println()
println("1.3 Symbolic computation of integral of Gb_ξ...")
@time begin
    tmp_Gb2 = sympy.diff(Gb,ξ).subs(ξ,0).subs(z,zp)
    inner_Gb2_integral = sympy.simplify(sympy.integrate(tmp_Gb2, (zp, ξ, 0)))
end

println("1.3 Done.")
# display(inner_Gb2_integral)

#### Now compute $G_{ \mathfrak{u}, b_\xi} = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi $

In [ ]:
# function sympy_istrue(x)
#     return string(x) == "True"
# end

# function sympy_isfalse(x)
#     return string(x) == "False"
# end

# function sympy_has(expr, var)
#     return sympy_istrue(expr.has(var))
# end

# function sympy_iszero(expr)
#     return sympy_istrue(sympy.Eq(expr, 0))
# end

# function sympy_boolprop(expr, prop::Symbol)
#     try
#         return Bool(getproperty(expr, prop))
#     catch
#         return false
#     end
# end

# function sympy_terms(expr)
#     # Return additive terms of expr.
#     # If expr is an Add, use its args. Otherwise expr is a single term.
#     if sympy_boolprop(expr, :is_Add)
#         return collect(expr.args)
#         # alternatively:
#         # return collect(expr.as_ordered_terms())
#     else
#         return [expr]
#     end
# end

# function is_sympy_exp(expr)
#     return startswith(string(expr), "exp(")
# end

# function fast_exp_integrate_definite(expr, var, lower, upper)
#     expr_expanded = sympy.expand(sympy.powsimp(expr, force=true))

#     terms = sympy_terms(expr_expanded)

#     total = sympy.S(0)

#     println("    Number of additive terms: ", length(terms))

#     for term in terms
#         term = sympy.powsimp(term, force=true)

#         # Split term into var-independent coefficient and var-dependent part
#         coeff, dep = term.as_independent(var, as_Add=false)

#         # Constant with respect to var
#         if !sympy_has(dep, var)
#             total += term * (upper - lower)
#             continue
#         end

#         dep = sympy.powsimp(dep, force=true)

#         if is_sympy_exp(dep)
#             exponent = dep.args[1]
#             slope = sympy.diff(exponent, var)

#             # exponent must be linear in var
#             if !sympy_has(slope, var)
#                 if sympy_iszero(slope)
#                     total += term * (upper - lower)
#                 else
#                     total += coeff * (
#                         sympy.exp(exponent.subs(var, upper)) -
#                         sympy.exp(exponent.subs(var, lower))
#                     ) / slope
#                 end
#             else
#                 println("    Falling back on nonlinear exponent term:")
#                 println("    ", term)
#                 total += sympy.integrate(term, (var, lower, upper))
#             end
#         else
#             println("    Falling back on non-single-exp term:")
#             println("    dep = ", dep)
#             total += sympy.integrate(term, (var, lower, upper))
#         end
#     end

#     return total
# end

# function integrate_Guv_times_nonpiecewise_fast_exp(Guv, inner_Gb2_integral)
#     println("  Starting split termwise exponential integration.")

#     expr = Guv * inner_Gb2_integral

#     ξs = get_sympy_symbol(expr, "ξ")
#     zs = get_sympy_symbol(expr, "z")
#     Hs = get_sympy_symbol(expr, "H")

#     Guv_branch1 = Guv.args[1].args[1]
#     Guv_branch2 = Guv.args[2].args[1]

#     println("  Lower interval [-H, z]...")
#     I_low = fast_exp_integrate_definite(
#         Guv_branch2 * inner_Gb2_integral,
#         ξs,
#         -Hs,
#         zs,
#     )

#     println("  Upper interval [z, 0]...")
#     I_high = fast_exp_integrate_definite(
#         Guv_branch1 * inner_Gb2_integral,
#         ξs,
#         zs,
#         0,
#     )

#     println("  Combining without heavy simplification.")
#     return I_low + I_high
# end

In [ ]:
println()
println("1.4 Symbolic computation of Guvb_ξ...")
@time begin
    Guvb_xi = sympy.integrate(sympy.expand(Guv * inner_Gb2_integral), (ξ, -H, 0))
    max_obj = sympy.Max(-H,z)
    Guvb_xi = Guvb_xi.subs(max_obj, z)
    Guvb_xi = sympy.factor(Guvb_xi.args[1].args[1])
end

println("1.4 Done.")
# display(Guvb_xi)

#### Compute $\partial_y G_{\mathfrak{u}}$

In [ ]:
println()
println("1.5 Symbolic computation of Guv_y...")
@time begin
    Guv_φ = sympy.diff(Guv,φ) 
    φ_eqn = sqrt(im * f(y) / (ν * Ek))
    φ_derivative = diff(φ_eqn,y).subs(diff(f(y),y),β)
    Latex_φ_derivative = β/ (2 * f(y)) * φ_eqn
    @assert Latex_φ_derivative == φ_derivative
    Guv_y = Guv_φ * φ_derivative
end

println("1.5 Done and Latex formula for φ derivative passes.")
# display(Guv_y)

#### Now compute $G_{ \mathfrak{u}_y, b} = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z';\xi)  \; dz'  \; d \xi $

In [ ]:
println()
println("1.6 Symbolic computation of the integral of ∂y Guv and Gb...")

# @time begin
#     Guv_branch1 = Latex_φ_derivative * Guv_φ.args[1].args[1]                    # z ≤ ξ
#     Guv_branch2 = Latex_φ_derivative * Guv_φ.args[2].args[1]                    # z ≥ ξ
#     Guv_y_b = integrate_Guv_Gb(Guv_branch1, Guv_branch2, inner_Gb_integral)
# end

# println("1.6 Symbolic computation of the integral of ∂y Guv and Gb completed.")
println("1.6 Skipped.")
# # display(Guv_y_b)

#### Now compute $G_{ \mathfrak{u}_y, b_\xi} = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi $

#### Convert `sympy` objects to `Symbolics` and save them to disk

In [ ]:
# Safe access to SymPy boolean properties such as is_Add, is_Mul, is_Pow
function sympy_boolprop(expr, prop::Symbol)
    try
        return Bool(getproperty(expr, prop))
    catch
        return false
    end
end

# True only if expr itself is a Piecewise
is_sympy_piecewise(expr) = startswith(string(expr), "Piecewise(")

# Convert a SymPy Piecewise with 2 branches using an externally supplied Symbolics condition
function sympy_piecewise_to_symbolics(P, condsym)
    branches = P.args
    @assert length(branches) == 2 "Expected exactly 2 branches in Piecewise"

    b1 = branches[1]
    b2 = branches[2]

    # Each branch is a SymPy Tuple-like object:
    # b.args[1] = expression
    # b.args[2] = condition
    expr1_sympy = b1.args[1]
    expr2_sympy = b2.args[1]

    expr1 = sympy_to_symbolics_with_piecewise(expr1_sympy, condsym)
    expr2 = sympy_to_symbolics_with_piecewise(expr2_sympy, condsym)

    return ifelse(condsym, expr1, expr2)
end

# Recursive converter
function sympy_to_symbolics_with_piecewise(expr, condsym)

    # Case 1: expr itself is Piecewise
    if is_sympy_piecewise(expr)
        return sympy_piecewise_to_symbolics(expr, condsym)
    end

    # Case 2: expr is a sum
    if sympy_boolprop(expr, :is_Add)
        out = 0
        for a in expr.args
            out += sympy_to_symbolics_with_piecewise(a, condsym)
        end
        return out
    end

    # Case 3: expr is a product
    if sympy_boolprop(expr, :is_Mul)
        out = 1
        for a in expr.args
            out *= sympy_to_symbolics_with_piecewise(a, condsym)
        end
        return out
    end

    # Case 4: expr is a power
    if sympy_boolprop(expr, :is_Pow)
        base = sympy_to_symbolics_with_piecewise(expr.args[1], condsym)
        expo = sympy_to_symbolics_with_piecewise(expr.args[2], condsym)
        return base ^ expo
    end

    # Important safety check:
    # If we get here and the expression still contains Piecewise,
    # we must not send it to sympy_to_symbolics.
    if occursin("Piecewise", string(expr))
        error("""
        Expression contains Piecewise but was not decomposed.

        expr = $expr

        args = $(expr.args)
        """)
    end

    # Case 5: ordinary SymPy expression
    return sympy_to_symbolics(expr, [])
end

In [ ]:
println()
println("1.7 Expressions conversion to Symbolics and save to file...")

@variables z ξ ξp
symbolics_inner_Gb2_integral = sympy_to_symbolics(inner_Gb2_integral, [])

symbolics_Guvb               = sympy_to_symbolics_with_piecewise(Guvb, z <= ξp)
symbolics_inner_Gb_integral  = sympy_to_symbolics_with_piecewise(inner_Gb_integral, ξ >= ξp)
symbolics_Gb                 = sympy_to_symbolics_with_piecewise(Gb, z <= ξ)
symbolics_Guv                = sympy_to_symbolics_with_piecewise(Guv, z <= ξ)
symbolics_Guvb_xi            = sympy_to_symbolics_with_piecewise(Guvb_xi, z <= ξp)
symbolics_Guv_y              = sympy_to_symbolics_with_piecewise(Guv_y, z <= ξ)



@save "symexpr.jld2" symbolics_Guvb symbolics_Gb symbolics_Guv symbolics_inner_Gb_integral symbolics_Guvb_xi symbolics_inner_Gb2_integral symbolics_Guv_y
println("1.7 Done and saved to: symexpr.jld2")
println("Check these expressions by running SymbolicsTesting.jl in a fresh Julia REPL.")
println("Plot the Guvb expression by running Plot_Guvb.ipynb.")